# ICT Trading Bot — Exploratory Notebook

Quick walkthrough of the framework: load data, run detections, generate signals, backtest, inspect features.

Run from the project root with the package importable on `sys.path`.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load configuration

In [ ]:
def load_yaml(name):
    with open(ROOT / 'config' / name) as f:
        return yaml.safe_load(f)

detection_cfg = load_yaml('detection_config.yaml')
strategy_cfg  = load_yaml('strategy_config.yaml')
model_cfg     = load_yaml('model_config.yaml')
risk_cfg      = load_yaml('risk_config.yaml')

detection_cfg

## 2. Load market data

Drop a CSV with columns `time, open, high, low, close, volume` into `data/` first, e.g. `data/EURUSD_M15.csv`.

In [ ]:
from src.utils.data_loader import load_csv

df = load_csv(ROOT / 'data' / 'EURUSD_M15.csv')
df.tail()

## 3. Run detections

In [ ]:
from src.detection.fvg         import detect_fvg
from src.detection.orderblock  import detect_order_blocks
from src.detection.liquidity   import detect_liquidity_sweeps
from src.detection.structure   import detect_bos, detect_choch
from src.detection.session     import add_session_features

fvgs    = detect_fvg(df, detection_cfg)
obs     = detect_order_blocks(df, detection_cfg)
sweeps  = detect_liquidity_sweeps(df, detection_cfg)
bos     = detect_bos(df, detection_cfg)
choch   = detect_choch(df, detection_cfg)
df_sess = add_session_features(df, detection_cfg)

len(fvgs), len(obs), len(sweeps), len(bos), len(choch)

## 4. Build features

In [ ]:
from src.features.builder import build_feature_pipeline, FEATURE_COLUMNS

X = build_feature_pipeline(
    df_sess, detections={'fvg': fvgs, 'ob': obs, 'sweeps': sweeps, 'bos': bos, 'choch': choch}, cfg=detection_cfg
)
print(X.shape, '— expected cols:', len(FEATURE_COLUMNS))
X.head()

## 5. Quick backtest

In [ ]:
from src.strategy.rule_based import generate_signals, signals_to_setups
from src.backtest.engine     import run_backtest
from src.backtest.metrics    import compute_metrics

signals = generate_signals(df_sess, {'fvg': fvgs, 'ob': obs, 'sweeps': sweeps, 'bos': bos, 'choch': choch}, strategy_cfg)
result  = run_backtest(df_sess, signals, strategy_cfg, risk_cfg)
metrics = compute_metrics(result)
metrics

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(result.equity_curve)
plt.title('Equity curve')
plt.ylabel('Equity')
plt.grid(alpha=0.3)
plt.show()